In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
from rotation.solver import initialize, solve_mixing
from rotation.symbolic import MASS, MIX, ANGLE

In [ ]:
# Mass info from online PDG (in GeV)
# https://pdglive.lbl.gov/Particle.action?node=S009
# https://pdglive.lbl.gov/Particle.action?node=S014
PDG = {
    MASS["pi"]: 0.1349768,
    MASS["eta"]: 0.547862,
}

In [ ]:
argument_order, system_of_equations, _ = initialize()

known = {
    MASS["pi"]: PDG[MASS["pi"]],
    MASS["eta"]: PDG[MASS["eta"]],
    ANGLE["alpha"]: np.pi/4,
    ANGLE["beta"]: np.pi/4,
    ANGLE["gamma"]: np.pi/4,
}

unknown = {
    MASS["phi"]: 1,
    MIX["phi"]: 1,
    MIX["pi"]: PDG[MASS["pi"]],
    MIX["eta"]: PDG[MASS["eta"]],
    MIX["1x2"]: 1,
    MIX["1x1"]: 1,
}

In [ ]:
x_label = ANGLE["gamma"]

solutions = []
for x in np.linspace(0,2*np.pi,  num=151):
    known[x_label] = x
    try:
        solved = solve_mixing(
            system_of_equations,
            argument_order,
            unknown,
            known,
            tolerance=1e-1,
        )
        solutions.append([known.copy(), solved.copy()])
    except ValueError as e:
        pass

In [ ]:
xs = np.array([sol[0][x_label] for sol in solutions])
# Reorganize solutions to be arrays of each unknown
data = {}
for sol in solutions:
    for y_label, data_point in sol[1].items():
        try:
            data[y_label].append(data_point)
        except KeyError:
            data[y_label] = [data_point]
for y_label, ys in data.items():
    data[y_label] = np.array(ys)
    data[y_label] = np.ma.array(ys, mask=data[y_label] < 0.0)

# Mask out all negatives
mask = np.zeros(list(data.values())[0].shape, dtype=bool)
for ys in data.values():
    mask = np.logical_or(mask, ys.mask)

# plotting
fig, axs = plt.subplots(3,1, figsize=(10,18))
for i, ax in enumerate(axs):
    for y_label, ys in data.items():
        match i:
            case 0:
                ax.plot(xs, ys.data, label=str(y_label), linewidth=1, marker=".")
            case 1:
                ax.scatter(xs, ys, label=str(y_label), linewidth=1, marker=".")
            case 2:
                ax.scatter(xs[~mask], ys.data[~mask], label=str(y_label), linewidth=1, marker=".")
    ax.set_xlabel(x_label)
    ax.set_ylabel("Mass [GeV]")
    title = []
    for k, v in solutions[0][0].items():
        if k is not x_label:
            title.append(f"{k}={v:.2e}")
    ax.set_title(", ".join(title))
    ax.legend()
    ax.grid(True)
    ax.set_xlim(xs.min(),xs.max())
    ax.set_ylim(-0.35, 0.65)
    fig.show()